# Module 8 — Testing (pytest) + Logging/Monitoring
Exam domain: **Security, Governance, Monitoring & Testing**

Runs standalone in Google Colab — no Databricks account needed.

In [ ]:
!pip install -q pyspark==3.5.1 delta-spark==3.2.0 pytest

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (SparkSession.builder
    .appName("Module8-Testing")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

## The pipeline under test
Reusing the pure functions from Module 3 — this is exactly why we wrote them as
input-DataFrame-in, output-DataFrame-out functions instead of a single script.

In [ ]:
def clean_orders(orders_df, min_order_date="2024-01-01"):
    return (orders_df
        .filter(F.col("amount") > 0)
        .filter(F.col("order_date") >= F.lit(min_order_date)))

def enrich_with_customer(orders_df, customers_df):
    return orders_df.join(customers_df, on="customer_id", how="left")

## Write pipeline_transforms.py and test_pipeline_transforms.py to disk
`pytest` needs real files; write them out, then run pytest from the shell.

In [ ]:
transforms_code = '''
from pyspark.sql import functions as F

def clean_orders(orders_df, min_order_date="2024-01-01"):
    return (orders_df
        .filter(F.col("amount") > 0)
        .filter(F.col("order_date") >= F.lit(min_order_date)))

def enrich_with_customer(orders_df, customers_df):
    return orders_df.join(customers_df, on="customer_id", how="left")
'''
with open("/content/pipeline_transforms.py", "w") as f:
    f.write(transforms_code)

In [ ]:
test_code = '''
import pytest
from pyspark.sql import SparkSession
from pipeline_transforms import clean_orders, enrich_with_customer

@pytest.fixture(scope="module")
def spark():
    return SparkSession.builder.appName("pytest").master("local[2]").getOrCreate()

def test_clean_orders_drops_negative_amount(spark):
    df = spark.createDataFrame([(1, "2024-02-01", -5.0)], ["order_id", "order_date", "amount"])
    result = clean_orders(df)
    assert result.count() == 0

def test_clean_orders_respects_min_date(spark):
    df = spark.createDataFrame(
        [(1, "2023-12-01", 10.0), (2, "2024-02-01", 10.0)],
        ["order_id", "order_date", "amount"])
    result = clean_orders(df, min_order_date="2024-01-01")
    assert result.count() == 1

def test_enrich_with_customer_left_join_keeps_unmatched(spark):
    orders = spark.createDataFrame([(1, 99)], ["order_id", "customer_id"])
    customers = spark.createDataFrame([(1, "Alice")], ["customer_id", "name"])
    result = enrich_with_customer(orders, customers)
    assert result.count() == 1
    assert result.filter("name IS NULL").count() == 1  # customer_id 99 has no match
'''
with open("/content/test_pipeline_transforms.py", "w") as f:
    f.write(test_code)

In [ ]:
!cd /content && python -m pytest test_pipeline_transforms.py -v

## Logging
Structured logging (as opposed to scattered `print()` calls) is what lets you
grep/filter pipeline runs in production.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("module8_pipeline")

def run_with_logging(orders_df, customers_df, min_order_date="2024-01-01"):
    logger.info("Pipeline started, min_order_date=%s", min_order_date)
    clean = clean_orders(orders_df, min_order_date)
    logger.info("clean_orders: %d -> %d rows", orders_df.count(), clean.count())
    enriched = enrich_with_customer(clean, customers_df)
    unmatched = enriched.filter("name IS NULL").count()
    if unmatched > 0:
        logger.warning("%d rows have no matching customer", unmatched)
    logger.info("Pipeline finished")
    return enriched

orders = spark.createDataFrame([(1, 1, "2024-02-01", 10.0), (2, 99, "2024-02-02", 5.0)],
                                ["order_id", "customer_id", "order_date", "amount"])
customers = spark.createDataFrame([(1, "Alice")], ["customer_id", "name"])
run_with_logging(orders, customers).show()

## What "monitoring" adds on top of logs
Logs answer "what happened in this one run." Monitoring answers "is the system
healthy over time": row-count trend anomalies, job duration trend, failure
rate/alerting — typically built from job run history + these same log lines
shipped to a central store (see the Databricks version for the native
equivalents).